# v2 overhead benchmark — analysis

Reads `bench/overhead_matrix.csv` produced by `bench/build_overhead_matrix.py`
and visualizes the v2 metrics:

- **Wall-clock overhead %** = `(t_traceml - t_baseline) / t_baseline`
- **Per-step median (ms)** — only available in `traceml_run` mode
- **Peak GPU memory delta (MB)** = `peak_gpu_traceml - peak_gpu_baseline`
- **Peak host RSS delta (MB)** = `peak_rss_traceml - peak_rss_baseline`

Trial counts are `n_trials` per (workload, mode); reported numbers are
mean ± std across trials. The v2 doc recommends N=5 for routine releases.

## Reading the charts

**Overhead % is only meaningful for long-enough workloads.** Short fixtures
(<60 s wall) have inflated overhead because TraceML's ~10 s startup cost
swamps the runtime. The v2 doc warns about this with its "drop first 100
iter or 5s" warmup rule. Use `bert_agnews` (~5 min/trial) and similarly
long workloads as the headline numbers; treat short workloads as
integration tests, not benchmarks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CSV = Path('/teamspace/studios/this_studio/traceml/bench/overhead_matrix.csv')
df = pd.read_csv(CSV)
df

## Derive overhead % and deltas

Pivot to (workload x metric) so baseline and traceml_run sit side-by-side,
then compute the v2 metrics.

In [ ]:
wide = df.pivot_table(
    index='workload',
    columns='mode',
    values=['wall_s_mean', 'wall_s_std',
            'peak_gpu_mem_gb_mean', 'peak_gpu_mem_gb_std',
            'peak_rss_gb_mean', 'peak_rss_gb_std',
            'step_avg_ms_mean', 'step_avg_ms_std',
            'n_trials'],
)

out = pd.DataFrame(index=wide.index)
out['n_baseline'] = wide['n_trials']['baseline']
out['n_traceml'] = wide['n_trials']['traceml_run']
out['wall_baseline_s'] = wide['wall_s_mean']['baseline']
out['wall_traceml_s'] = wide['wall_s_mean']['traceml_run']
out['wall_baseline_std'] = wide['wall_s_std']['baseline']
out['wall_traceml_std'] = wide['wall_s_std']['traceml_run']
out['overhead_pct'] = (
    100 * (out['wall_traceml_s'] - out['wall_baseline_s'])
    / out['wall_baseline_s']
)
out['step_avg_ms'] = wide['step_avg_ms_mean']['traceml_run']
out['step_avg_ms_std'] = wide['step_avg_ms_std']['traceml_run']
out['peak_gpu_baseline_gb'] = wide['peak_gpu_mem_gb_mean']['baseline']
out['peak_gpu_traceml_gb'] = wide['peak_gpu_mem_gb_mean']['traceml_run']
out['peak_gpu_delta_mb'] = (
    1000 * (out['peak_gpu_traceml_gb'] - out['peak_gpu_baseline_gb'])
)
out['peak_rss_baseline_gb'] = wide['peak_rss_gb_mean']['baseline']
out['peak_rss_traceml_gb'] = wide['peak_rss_gb_mean']['traceml_run']
out['peak_rss_delta_mb'] = (
    1000 * (out['peak_rss_traceml_gb'] - out['peak_rss_baseline_gb'])
)
out.round(3)

## Wall-clock overhead % per workload

The headline v2 metric. Bars annotated with absolute seconds (helps
judge whether a small % is meaningful — +5% on a 1s run is 50ms, vs
+5% on a 5min run is 15s).

In [ ]:
valid = out.dropna(subset=['overhead_pct']).sort_values('wall_traceml_s')
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(valid))
colors = [
    '#5A9BD4' if w >= 60 else '#D4A55A'  # blue = long, amber = short
    for w in valid['wall_traceml_s']
]
ax.bar(x, valid['overhead_pct'], color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(7, color='red', linewidth=0.7, linestyle='--',
           label='v2 release-gate threshold (+7%)')
ax.set_xticks(x)
ax.set_xticklabels(valid.index, rotation=30, ha='right')
ax.set_ylabel('Overhead % (traceml_run vs baseline)')
ax.set_title('Wall-clock overhead by workload  '
             '(blue = >=60s wall, amber = setup-dominated)')
for i, (w, ov) in enumerate(zip(valid['wall_traceml_s'], valid['overhead_pct'])):
    ax.text(i, ov, f'{ov:+.2f}%\n({w:.0f}s)',
            ha='center', va='bottom' if ov > 0 else 'top', fontsize=8)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Wall-time: baseline vs traceml_run side-by-side

Per-workload paired bars. Error bars are ± std across trials. Helps see
whether the gap between baseline and traceml_run is within trial variance
(noise) or above it (signal).

In [ ]:
valid = out.dropna(subset=['wall_baseline_s', 'wall_traceml_s'])
valid = valid.sort_values('wall_traceml_s')
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(valid))
w = 0.38
ax.bar(x - w/2, valid['wall_baseline_s'], w,
       yerr=valid['wall_baseline_std'].fillna(0),
       label='baseline', color='#888888', capsize=3)
ax.bar(x + w/2, valid['wall_traceml_s'], w,
       yerr=valid['wall_traceml_std'].fillna(0),
       label='traceml_run', color='#5A9BD4', capsize=3)
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(valid.index, rotation=30, ha='right')
ax.set_ylabel('wall_s (log scale)')
ax.set_title('Wall time per trial — baseline vs traceml_run (mean +/- std)')
ax.legend()
ax.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Peak GPU memory delta

TraceML's instrumentation does itself allocate some GPU memory (sampler
buffers, hook activations). v2 doc reports this as a delta vs baseline
in MB. Negative values would mean the baseline allocated more — typically
doesn't happen but worth knowing.

In [ ]:
valid = out.dropna(subset=['peak_gpu_delta_mb']).sort_values('peak_gpu_traceml_gb')
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(valid))
ax.bar(x, valid['peak_gpu_delta_mb'], color='#54A24B',
       edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(valid.index, rotation=30, ha='right')
ax.set_ylabel('Peak GPU memory delta (MB)')
ax.set_title('Peak GPU memory: traceml_run minus baseline')
for i, (d, b, t) in enumerate(zip(
        valid['peak_gpu_delta_mb'],
        valid['peak_gpu_baseline_gb'],
        valid['peak_gpu_traceml_gb'])):
    ax.text(i, d, f'{d:+.0f} MB\n({b:.2f} -> {t:.2f} GB)',
            ha='center', va='bottom' if d > 0 else 'top', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Peak host RSS delta

Same shape as the GPU memory delta but on the CPU side. TraceML runs an
out-of-process aggregator that adds RSS — measurable here.

In [ ]:
valid = out.dropna(subset=['peak_rss_delta_mb']).sort_values('peak_rss_traceml_gb')
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(valid))
ax.bar(x, valid['peak_rss_delta_mb'], color='#B279A2',
       edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(valid.index, rotation=30, ha='right')
ax.set_ylabel('Peak host RSS delta (MB)')
ax.set_title('Peak host RSS: traceml_run minus baseline')
for i, (d, b, t) in enumerate(zip(
        valid['peak_rss_delta_mb'],
        valid['peak_rss_baseline_gb'],
        valid['peak_rss_traceml_gb'])):
    ax.text(i, d, f'{d:+.0f} MB\n({b:.2f} -> {t:.2f} GB)',
            ha='center', va='bottom' if d > 0 else 'top', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Headline summary table

Sorted by absolute traceml_run wall — long workloads at top (use these for
headline overhead %), short workloads at bottom (treat as integration
tests). Threshold from v2 doc: any workload with overhead > +7% relative
to the previous release flags a regression.

In [ ]:
summary = out[[
    'n_traceml',
    'wall_baseline_s', 'wall_traceml_s', 'overhead_pct',
    'step_avg_ms',
    'peak_gpu_delta_mb', 'peak_rss_delta_mb',
]].copy()
summary = summary.sort_values('wall_traceml_s', ascending=False)
summary.round({
    'wall_baseline_s': 1, 'wall_traceml_s': 1,
    'overhead_pct': 2, 'step_avg_ms': 2,
    'peak_gpu_delta_mb': 0, 'peak_rss_delta_mb': 0,
})